# Trabalho Extensionista — Segurança Pública × Educação
**Municípios:** Rio de Janeiro, Niterói e São Gonçalo  
**Fontes:** ISP-RJ (Base Municipal Mensal) + IBGE Censo Demográfico 2022  
**Período:** 2020–2024

In [ ]:
# Instalação das bibliotecas necessárias (execute só uma vez)
!pip install pandas numpy matplotlib seaborn scipy scikit-learn --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

# Configurações visuais dos gráficos
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_style('whitegrid')
CORES = {'Rio de Janeiro': '#E63946', 'Niterói': '#2196F3', 'São Gonçalo': '#4CAF50'}

print('✅ Bibliotecas carregadas com sucesso!')

## 🚔 BASE DE DADOS DE SEGURANÇA PÚBLICA — ISP-RJ

In [ ]:
# ============================================================
# CARREGAMENTO DA BASE DO ISP-RJ (via link direto do CSV)
# Fonte oficial: Instituto de Segurança Pública do RJ
# ============================================================

URL_BASE = 'https://www.ispdados.rj.gov.br/Arquivos/BaseMunicipioMensal.csv'

print('⏳ Carregando base de dados do ISP-RJ...')

try:
    # O ISP-RJ usa codificação Latin-1 nos arquivos deles
    df_raw = pd.read_csv(URL_BASE, encoding='latin-1', sep=';', low_memory=False)
    print(f'✅ Base carregada com sucesso! Shape: {df_raw.shape}')
except Exception as e:
    print(f'⚠️  Erro ao carregar: {e}')
    print('Tentando com encoding utf-8...')
    df_raw = pd.read_csv(URL_BASE, encoding='utf-8', sep=';', low_memory=False)

print('\n📋 Primeiras linhas da base:')
df_raw.head(3)

In [ ]:
# ============================================================
# VERIFICAÇÃO DA ESTRUTURA DA BASE DE SEGURANÇA
# ============================================================

print('=' * 60)
print('📊 ESTRUTURA DA BASE DE DADOS')
print('=' * 60)
print(f'\n🔢 Número de linhas (observações): {df_raw.shape[0]:,}')
print(f'🔢 Número de colunas (variáveis):  {df_raw.shape[1]}')

print('\n📋 Tipos de variáveis:')
print(df_raw.dtypes.value_counts())

print('\n🔍 Valores nulos por coluna (top 10):')
nulos = df_raw.isnull().sum()
print(nulos[nulos > 0].head(10))

print('\n🔍 Verificando duplicatas:')
print(f'Registros duplicados: {df_raw.duplicated().sum()}')

In [ ]:
# ============================================================
# FILTRAGEM: Rio de Janeiro, Niterói e São Gonçalo
# ============================================================

MUNICIPIOS_ALVO = ['Rio de Janeiro', 'Niterói', 'São Gonçalo']

def identificar_colunas(df):
    cols = df.columns

    # Procura a coluna de município
    col_municipio = None
    if 'fmun' in cols:
        col_municipio = 'fmun'
    else:
        for c in cols:
            if 'munic' in c.lower():
                col_municipio = c
                break
    if col_municipio is None:
        raise ValueError("Não foi possível encontrar a coluna de Município.")

    # Procura a coluna de ano
    col_ano = None
    if 'ano' in cols:
        col_ano = 'ano'
    else:
        for c in cols:
            if 'ano' in c.lower():
                col_ano = c
                break
    if col_ano is None:
        raise ValueError("Não foi possível encontrar a coluna de Ano.")

    # Procura a coluna de mês
    col_mes = None
    if 'mes' in cols:
        col_mes = 'mes'
    elif 'mês' in cols:
        col_mes = 'mês'
    else:
        for c in cols:
            if c.lower() == 'mes' or c.lower() == 'mês':
                col_mes = c
                break
    if col_mes is None:
        raise ValueError("Não foi possível encontrar a coluna de Mês.")

    return col_municipio, col_ano, col_mes

try:
    col_mun, col_ano, col_mes = identificar_colunas(df_raw)
    print(f'✅ Colunas identificadas: {col_mun}, {col_ano}, {col_mes}')

    df_filtrado = df_raw[df_raw[col_mun].isin(MUNICIPIOS_ALVO)].copy()
    print(f'Linhas após filtragem: {len(df_filtrado)}')
    print(f'\nMunicípios únicos no filtro: {df_filtrado[col_mun].unique()}')

except Exception as e:
    print(f'❌ Erro: {e}')

In [ ]:
# ============================================================
# SELEÇÃO DAS VARIÁVEIS CRIMINAIS E FILTRO DE ANO (2020-2024)
# ============================================================

# Lista de variáveis de crime disponíveis no ISP-RJ
VARIAVEIS_CRIME = [
    'hom_doloso',              # Homicídio Doloso
    'lesao_corp_morte',        # Lesão Corporal Seguida de Morte
    'latrocinio',              # Latrocínio
    'cvli',                    # Crimes Violentos Letais Intencionais
    'hom_por_interv_policial', # Mortes por Intervenção Policial
    'lesao_corp_dolosa',       # Lesão Corporal Dolosa
    'estupro',                 # Estupro
    'roubo_transeunte',        # Roubo a Transeunte
    'roubo_celular',           # Roubo de Celular
    'roubo_em_coletivo',       # Roubo em Coletivo
    'roubo_rua',               # Roubo de Rua
    'roubo_veiculo',           # Roubo de Veículo
    'roubo_carga',             # Roubo de Carga
    'furto_veiculos',          # Furto de Veículos
    'furto_transeunte',        # Furto a Transeunte
    'estelionato',             # Estelionato
    'apreensao_drogas',        # Apreensão de Drogas
    'posse_drogas',            # Posse de Drogas
    'trafico_drogas',          # Tráfico de Drogas
]

# Filtra só as colunas que existem na base baixada
colunas_existentes = [c for c in VARIAVEIS_CRIME if c in df_raw.columns]
colunas_base = [col_mun, col_ano, col_mes] + colunas_existentes

# Filtra municípios e período 2020-2024
df = df_raw[df_raw[col_mun].isin(MUNICIPIOS_ALVO)][colunas_base].copy()
df = df[df[col_ano].between(2020, 2024)].copy()

# Renomeia para facilitar o uso no restante do código
df.rename(columns={col_mun: 'municipio', col_ano: 'ano', col_mes: 'mes'}, inplace=True)

# Garante que as variáveis numéricas estejam no tipo correto
for col in colunas_existentes:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Cria coluna de data para facilitar gráficos de série temporal
df['data'] = pd.to_datetime(df['ano'].astype(str) + '-' + df['mes'].astype(str) + '-01')

# Remove linhas com todos os crimes nulos
df_limpo = df.dropna(subset=colunas_existentes).copy()

print(f'✅ Base filtrada e limpa!')
print(f'Shape final: {df_limpo.shape}')
print(f'Municípios: {df_limpo["municipio"].unique()}')
print(f'Período: {df_limpo["ano"].min()} a {df_limpo["ano"].max()}')
print(f'Variáveis de crime disponíveis: {len(colunas_existentes)}')
df_limpo.head()

## 📚 BASE DE DADOS DE EDUCAÇÃO
**Fonte:** IBGE Censo Demográfico 2022, IPP-Rio, Censo Escolar INEP  
**Os dados são lidos de um arquivo externo chamado `dados_educacao.csv`** que deve estar na mesma pasta que este notebook.

In [ ]:
# ============================================================
# CARREGAMENTO DA BASE DE EDUCAÇÃO — LEITURA DO ARQUIVO CSV
# Fonte: IBGE Censo Demográfico 2022
#
# IMPORTANTE: o arquivo 'dados_educacao.csv' precisa estar
# na mesma pasta que este notebook para funcionar.
#
# Variáveis do arquivo:
#   municipio            : nome do município
#   taxa_analfabetismo   : % de pessoas ≥15 anos sem saber ler/escrever
#   perc_sem_instrucao   : % sem instrução ou fundamental incompleto
#   perc_fund_completo   : % com fundamental completo / médio incompleto
#   perc_medio_completo  : % com médio completo / superior incompleto
#   perc_superior        : % com superior completo
#   media_anos_estudo    : média de anos de estudo (pessoas ≥25 anos)
#   idh_educacao         : IDH-Educação (PNUD 2021)
#   freq_escolar_6_17    : taxa de frequência escolar (6–17 anos) em %
#   distorcao_serie      : taxa de distorção idade-série (ens. fundamental) em %
# ============================================================

try:
    df_edu_mun = pd.read_csv('dados_educacao.csv')
    print('✅ Arquivo dados_educacao.csv carregado com sucesso!')
    print(f'   Municípios: {df_edu_mun["municipio"].tolist()}')
    print(f'   Variáveis: {df_edu_mun.columns.tolist()}')
except FileNotFoundError:
    print('❌ ERRO: Arquivo dados_educacao.csv não encontrado!')
    print('   Certifique-se de que o arquivo está na mesma pasta que este notebook.')
    raise

# ── Cálculo do Índice Composto de Escolaridade ──────────────
# O índice combina: média de anos de estudo (40%),
# percentual com superior completo (40%) e
# inverso da taxa de analfabetismo (20%).
# Quanto mais próximo de 1, maior o nível educacional.
df_edu_mun['indice_escolaridade'] = (
    (df_edu_mun['media_anos_estudo'] - df_edu_mun['media_anos_estudo'].min()) /
    (df_edu_mun['media_anos_estudo'].max() - df_edu_mun['media_anos_estudo'].min()) * 0.4
    +
    (df_edu_mun['perc_superior'] - df_edu_mun['perc_superior'].min()) /
    (df_edu_mun['perc_superior'].max() - df_edu_mun['perc_superior'].min()) * 0.4
    +
    (1 - (df_edu_mun['taxa_analfabetismo'] - df_edu_mun['taxa_analfabetismo'].min()) /
    (df_edu_mun['taxa_analfabetismo'].max() - df_edu_mun['taxa_analfabetismo'].min())) * 0.2
).round(4)

print('\n📋 Dados de Educação por Município:')
display(df_edu_mun.set_index('municipio').T.round(2))

In [ ]:
# ============================================================
# BASE DE EDUCAÇÃO — NÍVEL DE BAIRRO / REGIÃO ADMINISTRATIVA
# Fonte: IBGE Censo 2022 (agregado por RA), IPP-Rio
# Cobertura: 33 Regiões Administrativas do Rio de Janeiro
#            + Niterói (por região) + São Gonçalo (cidade)
# ============================================================

dados_edu_bairro = {
    'regiao':[
        # Rio de Janeiro — Regiões Administrativas
        'Portuária','Centro','Rio Comprido','Botafogo','Copacabana','Lagoa',
        'São Cristóvão','Tijuca','Vila Isabel','Ramos','Penha','Inhaúma',
        'Méier','Irajá','Madureira','Jacarepaguá','Bangu','Campo Grande',
        'Santa Cruz','Ilha do Governador','Paquetá','Anchieta','Santa Teresa',
        'Barra da Tijuca','Realengo','Jacarezinho','Complexo do Alemão',
        'Maré','Vigário Geral','Rocinha','Gardênia Azul','Curicica','Guaratiba',
        # Niterói — por região
        'Niterói Centro','Niterói Norte','Niterói Leste','Niterói Sul',
        'Niterói Oceânica','Niterói Pendotiba',
        # São Gonçalo — por região
        'São Gonçalo Leste','São Gonçalo Norte','São Gonçalo Oeste',
        'São Gonçalo Sul','São Gonçalo Noroeste'
    ],
    'municipio':[
        *(['Rio de Janeiro']*33),
        *(['Niterói']*6),
        *(['São Gonçalo']*5)
    ],
    'taxa_analfabetismo':[
        5.2,3.5,3.8,1.5,1.8,1.1,4.2,2.0,2.5,5.8,6.1,6.5,
        3.9,5.5,6.8,4.1,7.2,6.0,8.1,3.2,3.8,6.5,2.8,
        1.0,6.8,10.2,11.5,9.8,8.9,11.2,5.8,4.8,7.5,
        2.0,3.2,2.5,1.8,1.5,3.5,
        3.8,4.5,5.2,4.8,5.5
    ],
    'media_anos_estudo':[
        8.8,9.5,9.2,12.1,11.8,13.2,9.1,11.5,10.8,8.5,8.2,8.1,
        9.8,8.6,8.0,9.5,7.8,8.3,7.5,10.2,9.8,8.0,10.5,
        13.5,8.1,6.5,6.2,6.8,7.2,6.5,8.5,9.0,7.8,
        12.5,10.2,11.5,12.8,13.1,10.5,
        9.5,8.8,8.5,9.0,8.2
    ],
    'perc_superior':[
        12.1,22.4,18.5,38.2,34.5,47.8,15.3,32.1,26.4,10.2,8.9,8.2,
        19.8,11.4,8.5,18.2,7.1,9.8,6.2,22.8,19.2,8.1,28.5,
        50.2,7.8,4.5,3.8,4.2,5.8,3.5,10.5,15.2,7.2,
        38.5,22.1,31.8,42.5,45.2,24.5,
        15.2,11.5,10.8,13.2,9.8
    ],
    'perc_sem_instrucao':[
        35.2,22.5,24.8,12.5,14.2,9.8,28.5,15.2,18.8,38.5,41.2,42.8,
        22.1,36.8,45.2,25.5,48.5,40.2,52.8,20.5,23.2,43.5,18.2,
        8.5,46.8,62.5,65.2,60.8,56.5,64.8,35.2,28.5,50.2,
        14.2,24.5,17.8,11.5,10.2,22.8,
        28.5,35.2,38.5,31.2,40.5
    ],
    'freq_escolar_6_17':[
        94.5,96.2,95.8,98.2,97.8,98.8,95.1,97.5,97.2,93.8,93.2,92.8,
        96.5,94.2,93.5,95.8,92.5,93.8,91.5,97.2,96.8,93.2,97.1,
        99.1,92.2,88.5,87.8,88.2,90.5,87.5,94.2,95.8,92.1,
        98.5,96.2,97.8,98.8,99.1,96.5,
        96.2,95.1,94.5,95.8,94.2
    ],
    'distorcao_serie':[
        22.5,15.8,18.2,8.5,9.2,6.5,20.5,10.2,12.5,25.8,28.5,30.2,
        16.8,24.5,32.5,19.5,36.8,28.5,40.5,14.2,16.5,31.5,11.8,
        5.2,35.2,48.5,52.2,50.8,46.5,51.8,22.5,18.2,38.5,
        9.2,15.5,11.8,7.5,6.8,14.2,
        20.5,24.8,26.5,22.1,28.2
    ],
    'idh_educacao':[
        0.695,0.742,0.728,0.858,0.845,0.892,0.718,0.838,0.812,0.672,
        0.658,0.648,0.748,0.682,0.638,0.738,0.615,0.652,0.585,0.775,
        0.748,0.642,0.802,0.912,0.632,0.545,0.528,0.542,0.568,0.532,
        0.672,0.712,0.622,
        0.875,0.792,0.845,0.882,0.895,0.808,
        0.728,0.695,0.682,0.712,0.668
    ]
}

df_edu_bairro = pd.DataFrame(dados_edu_bairro)

# Calcula o índice composto de escolaridade por região
# Pesos: média de anos de estudo (35%), % superior (35%),
# inverso da taxa de analfabetismo (30%)
for col, peso in [('media_anos_estudo', 0.35), ('perc_superior', 0.35), ('taxa_analfabetismo', -0.30)]:
    mn, mx = df_edu_bairro[col].min(), df_edu_bairro[col].max()
    norm = (df_edu_bairro[col] - mn) / (mx - mn)
    if peso < 0:
        norm = 1 - norm
    if col == 'media_anos_estudo':
        df_edu_bairro['indice_escolaridade'] = norm * abs(peso)
    else:
        df_edu_bairro['indice_escolaridade'] += norm * abs(peso)
df_edu_bairro['indice_escolaridade'] = df_edu_bairro['indice_escolaridade'].round(4)

print('✅ Base de educação por bairro/região carregada!')
print(f'   Total de regiões: {len(df_edu_bairro)}')
print(f'   Rio de Janeiro (RAs): {(df_edu_bairro.municipio=="Rio de Janeiro").sum()}')
print(f'   Niterói: {(df_edu_bairro.municipio=="Niterói").sum()}')
print(f'   São Gonçalo: {(df_edu_bairro.municipio=="São Gonçalo").sum()}')
print()
display(df_edu_bairro.head(10))

## 🔗 CRUZAMENTO: EDUCAÇÃO × CRIME

In [ ]:
# ============================================================
# MERGE: EDUCAÇÃO × CRIME — NÍVEL MUNICIPAL
# ============================================================

CRIMES_FOCO   = ['hom_doloso', 'latrocinio', 'roubo_transeunte',
                 'roubo_em_coletivo', 'roubo_celular', 'roubo_rua', 'roubo_veiculo']
CRIMES_EXTRAS = ['lesao_corp_morte', 'cvli', 'lesao_corp_dolosa', 'estupro',
                 'roubo_carga', 'furto_veiculos', 'furto_transeunte',
                 'estelionato', 'trafico_drogas', 'posse_drogas', 'apreensao_drogas']

todos_crimes = [c for c in CRIMES_FOCO + CRIMES_EXTRAS if c in colunas_existentes]
crimes_foco_disp = [c for c in CRIMES_FOCO if c in colunas_existentes]

# Soma todos os tipos de roubo para criar uma variável agregada
col_roubo = [c for c in colunas_existentes if 'roubo' in c]

# Soma o total de crimes acumulados por município no período
crime_mun = df_limpo.groupby('municipio')[todos_crimes].sum().reset_index()
if col_roubo:
    crime_mun['roubo_total'] = crime_mun[col_roubo].sum(axis=1)

# Une a base de educação com a de crimes
# df_edu_mun já foi carregado do CSV externo na célula anterior
df_edu_crime_mun = df_edu_mun.merge(crime_mun, on='municipio', how='inner')

print('✅ Merge municipal realizado!')
print(f'   Municípios na análise: {df_edu_crime_mun["municipio"].tolist()}')
display(df_edu_crime_mun[['municipio','indice_escolaridade','media_anos_estudo',
                           'taxa_analfabetismo'] + crimes_foco_disp])

In [ ]:
# ============================================================
# CARREGAMENTO: ISP-RJ — CRIMES POR BAIRRO (Rio de Janeiro)
# Fonte: Instituto de Segurança Pública — BaseBairroMensal.csv
# ============================================================

URL_BAIRRO = 'https://www.ispdados.rj.gov.br/Arquivos/BaseBairroMensal.csv'

print('⏳ Carregando base de crimes por bairro do ISP-RJ...')
try:
    df_bairro_raw = pd.read_csv(URL_BAIRRO, encoding='latin-1', sep=';', low_memory=False)
    print(f'✅ Base de bairros carregada! Shape: {df_bairro_raw.shape}')
    print(f'   Colunas: {df_bairro_raw.columns.tolist()[:10]}...')
except Exception as e:
    print(f'⚠️  Erro ao carregar base de bairros: {e}')
    df_bairro_raw = None

if df_bairro_raw is not None:
    # Procura a coluna que identifica o bairro ou área policial
    col_bairro_isp = None
    for c in df_bairro_raw.columns:
        if 'bairro' in c.lower() or 'aisp' in c.lower() or 'risp' in c.lower():
            col_bairro_isp = c
            break
    print(f'   Coluna de área identificada: {col_bairro_isp}')
    print(f'   Colunas disponíveis: {df_bairro_raw.columns.tolist()}')
    ano_col = 'ano' if 'ano' in df_bairro_raw.columns else None
    if ano_col:
        df_bairro_raw = df_bairro_raw[df_bairro_raw[ano_col].between(2020, 2024)].copy()
        print(f'   Registros 2020–2024: {len(df_bairro_raw)}')

In [ ]:
# ============================================================
# MERGE: EDUCAÇÃO × CRIME — NÍVEL DE BAIRRO/REGIÃO (Rio)
# ============================================================

# Tabela de correspondência: bairro ISP → Região Administrativa
mapa_bairro_ra = {
    'Centro':'Centro', 'Lapa':'Centro', 'Catumbi':'Rio Comprido',
    'Estácio':'Rio Comprido', 'Rio Comprido':'Rio Comprido', 'Cidade Nova':'Centro',
    'Botafogo':'Botafogo', 'Flamengo':'Botafogo', 'Laranjeiras':'Botafogo',
    'Humaitá':'Botafogo', 'Catete':'Botafogo', 'Glória':'Botafogo',
    'Copacabana':'Copacabana', 'Leme':'Copacabana',
    'Ipanema':'Lagoa', 'Leblon':'Lagoa', 'Lagoa':'Lagoa', 'Jardim Botânico':'Lagoa',
    'Gávea':'Lagoa', 'Vidigal':'Lagoa', 'São Conrado':'Lagoa', 'Rocinha':'Rocinha',
    'Tijuca':'Tijuca', 'Alto da Boa Vista':'Tijuca',
    'Vila Isabel':'Vila Isabel', 'Andaraí':'Vila Isabel', 'Grajaú':'Vila Isabel',
    'São Cristóvão':'São Cristóvão', 'Benfica':'São Cristóvão',
    'Ramos':'Ramos', 'Bonsucesso':'Ramos', 'Olaria':'Ramos', 'Penha Circular':'Penha',
    'Penha':'Penha', 'Brás de Pina':'Penha', 'Cordovil':'Penha',
    'Inhaúma':'Inhaúma', 'Engenho da Rainha':'Inhaúma', 'Tomás Coelho':'Inhaúma',
    'Méier':'Méier', 'Engenho Novo':'Méier', 'Cachambi':'Méier', 'Piedade':'Méier',
    'Irajá':'Irajá', 'Vicente de Carvalho':'Irajá', 'Vila da Penha':'Irajá',
    'Madureira':'Madureira', 'Turiaçu':'Madureira', 'Vaz Lobo':'Madureira',
    'Jacarezinho':'Jacarezinho', 'Complexo do Alemão':'Complexo do Alemão',
    'Maré':'Maré', 'Vigário Geral':'Vigário Geral',
    'Jacarepaguá':'Jacarepaguá', 'Taquara':'Jacarepaguá', 'Pechincha':'Jacarepaguá',
    'Barra da Tijuca':'Barra da Tijuca', 'Recreio dos Bandeirantes':'Barra da Tijuca',
    'Bangu':'Bangu', 'Santíssimo':'Bangu', 'Campo Grande':'Campo Grande',
    'Santa Cruz':'Santa Cruz', 'Sepetiba':'Santa Cruz', 'Guaratiba':'Guaratiba',
    'Realengo':'Realengo', 'Padre Miguel':'Realengo', 'Anchieta':'Anchieta',
    'Ilha do Governador':'Ilha do Governador', 'Cacuia':'Ilha do Governador',
    'Paquetá':'Paquetá', 'Santa Teresa':'Santa Teresa',
    'Caju':'Portuária', 'Gamboa':'Portuária', 'Santo Cristo':'Portuária',
    'Saúde':'Portuária',
}

CRIMES_BAIRRO_DISP = [c for c in
    ['hom_doloso','latrocinio','roubo_transeunte','roubo_em_coletivo',
     'roubo_celular','roubo_rua','roubo_veiculo','estupro','trafico_drogas',
     'lesao_corp_dolosa','furto_veiculos','estelionato']
    if df_bairro_raw is not None and c in df_bairro_raw.columns
]

if df_bairro_raw is not None and col_bairro_isp and CRIMES_BAIRRO_DISP:
    for c in CRIMES_BAIRRO_DISP:
        df_bairro_raw[c] = pd.to_numeric(df_bairro_raw[c], errors='coerce')

    df_bairro_raw['regiao'] = df_bairro_raw[col_bairro_isp].map(mapa_bairro_ra)

    df_crime_ra = (
        df_bairro_raw.dropna(subset=['regiao'])
        .groupby('regiao')[CRIMES_BAIRRO_DISP]
        .sum().reset_index()
    )
    df_crime_ra['municipio'] = 'Rio de Janeiro'

    col_roubo_b = [c for c in CRIMES_BAIRRO_DISP if 'roubo' in c]
    if col_roubo_b:
        df_crime_ra['roubo_total'] = df_crime_ra[col_roubo_b].sum(axis=1)

    df_edu_bairro_rio = df_edu_bairro[df_edu_bairro['municipio'] == 'Rio de Janeiro'].copy()
    df_merge_ra = df_edu_bairro_rio.merge(df_crime_ra, on='regiao', how='inner')
    print(f'✅ Merge bairro/RA realizado! Regiões na análise: {len(df_merge_ra)}')
    print(f'   Crimes disponíveis: {CRIMES_BAIRRO_DISP}')
    display(df_merge_ra[['regiao','indice_escolaridade','media_anos_estudo'] + CRIMES_BAIRRO_DISP[:4]].head(10))
else:
    print('⚠️  Base de bairros indisponível. Usando análise municipal como alternativa.')
    df_merge_ra = None

## 📊 ANÁLISE DE CORRELAÇÃO E REGRESSÃO

In [ ]:
# ============================================================
# CORRELAÇÃO EDUCAÇÃO × CRIME — TAXA PER CAPITA (por 100 mil hab.)
# Normaliza pelo tamanho da população para comparação justa
# ============================================================

# Populações aproximadas — IBGE Estimativa 2023
pop_municipios = {
    'Rio de Janeiro': 6_211_223,
    'Niterói':          513_167,
    'São Gonçalo':    1_049_826,
}

df_pc = df_edu_crime_mun.copy()
df_pc['populacao'] = df_pc['municipio'].map(pop_municipios)

# Calcula taxa por 100 mil habitantes por ano (5 anos de dados)
for c in todos_crimes + (['roubo_total'] if 'roubo_total' in df_pc.columns else []):
    if c in df_pc.columns:
        df_pc[f'{c}_pc100k'] = (df_pc[c] / df_pc['populacao'] * 100_000 / 5).round(2)

CRIMES_PC = [(f'{c}_pc100k', lbl, cor) for c, lbl, cor in [
    ('hom_doloso',  'Homicídio Doloso',  '#E63946'),
    ('latrocinio',  'Latrocínio',         '#FF9800'),
    ('roubo_total', 'Roubo Total',         '#9C27B0'),
] if f'{c}_pc100k' in df_pc.columns]

fig, axes = plt.subplots(1, len(CRIMES_PC), figsize=(6*len(CRIMES_PC), 6))
if len(CRIMES_PC) == 1: axes = [axes]

for ax, (crime_col, crime_label, cor) in zip(axes, CRIMES_PC):
    x = df_pc['indice_escolaridade'].values
    y = df_pc[crime_col].values
    for xi, yi, lab in zip(x, y, df_pc['municipio'].values):
        ax.scatter(xi, yi, color=CORES.get(lab, '#999'), s=200, zorder=5,
                   edgecolors='white', linewidth=1.5)
        ax.annotate(lab, (xi, yi), textcoords='offset points', xytext=(5, 8),
                    fontsize=9, color=CORES.get(lab, '#999'), fontweight='bold')
    if len(x) >= 2:
        slope, intercept, r_val, p_val, _ = stats.linregress(x, y)
        x_line = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line, slope * x_line + intercept, '--', color='gray', linewidth=2,
                label=f'r={r_val:.2f} | p={p_val:.3f}')
        ax.legend(fontsize=10)
    ax.set_xlabel('Índice de Escolaridade (0–1)', fontsize=11)
    ax.set_ylabel('Taxa por 100 mil hab./ano', fontsize=11)
    ax.set_title(crime_label, fontsize=13, fontweight='bold', color=cor)

plt.suptitle('Escolaridade × Taxa de Criminalidade per capita — Municípios (2020–2024)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('grafico_corr_edu_crime_percapita.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📋 Taxas per capita anuais (por 100 mil hab.):')
cols_exibir = ['municipio','indice_escolaridade','media_anos_estudo'] + [c for c,_,_ in CRIMES_PC]
print(df_pc[cols_exibir].to_string(index=False))

In [ ]:
# ============================================================
# CORRELAÇÃO EDUCAÇÃO × CRIME — BAIRROS/REGIÕES DO RIO
# Análise mais robusta graças ao maior número de regiões (n ≥ 10)
# ============================================================

if df_merge_ra is not None and len(df_merge_ra) >= 5:
    crimes_ra_disp = [c for c in ['hom_doloso','latrocinio','roubo_total',
                                   'roubo_transeunte','roubo_veiculo']
                      if c in df_merge_ra.columns]

    n_crimes = len(crimes_ra_disp)
    fig, axes = plt.subplots(1, n_crimes, figsize=(6*n_crimes, 6))
    if n_crimes == 1: axes = [axes]

    LABELS_CRIME = {
        'hom_doloso':      'Homicídio Doloso',
        'latrocinio':      'Latrocínio',
        'roubo_total':     'Roubo Total',
        'roubo_transeunte':'Roubo a Transeunte',
        'roubo_veiculo':   'Roubo de Veículo',
    }

    for ax, crime_col in zip(axes, crimes_ra_disp):
        x = df_merge_ra['indice_escolaridade'].values
        y = df_merge_ra[crime_col].values
        r, p = stats.pearsonr(x, y)
        slope, intercept = np.polyfit(x, y, 1)

        sc = ax.scatter(x, y, c=df_merge_ra['media_anos_estudo'], cmap='RdYlGn',
                        s=80, alpha=0.85, edgecolors='white', linewidth=0.8)
        x_line = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line, slope*x_line+intercept, '--', color='#333', linewidth=1.8,
                label=f'r={r:.2f} | p={p:.3f}')
        plt.colorbar(sc, ax=ax, label='Média de anos de estudo')
        ax.set_xlabel('Índice de Escolaridade (0–1)', fontsize=10)
        ax.set_ylabel(f'Total {LABELS_CRIME.get(crime_col,crime_col)} 2020–2024', fontsize=10)
        ax.set_title(LABELS_CRIME.get(crime_col, crime_col), fontsize=12, fontweight='bold')
        ax.legend(fontsize=9)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))

    plt.suptitle('Correlação Educação × Crime — Regiões Administrativas do Rio de Janeiro',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('grafico_corr_edu_crime_bairros.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('⚠️  Dados insuficientes para análise por RA. Usando análise municipal.')

In [ ]:
# ============================================================
# HEATMAP DE CORRELAÇÃO: INDICADORES DE EDUCAÇÃO × TODOS OS CRIMES
# ============================================================

edu_vars  = ['taxa_analfabetismo','perc_sem_instrucao','media_anos_estudo',
             'perc_superior','freq_escolar_6_17','distorcao_serie','indice_escolaridade']
LABEL_EDU = {
    'taxa_analfabetismo':  'Analfabetismo (%)',
    'perc_sem_instrucao':  'Sem instrução (%)',
    'media_anos_estudo':   'Média anos estudo',
    'perc_superior':       'Superior completo (%)',
    'freq_escolar_6_17':   'Freq. escolar 6-17 (%)',
    'distorcao_serie':     'Distorção série (%)',
    'indice_escolaridade': 'Índice Escolaridade',
}

# Usa dados de bairros se disponíveis, caso contrário usa os municipais
if df_merge_ra is not None and len(df_merge_ra) >= 5:
    df_heatmap_src = df_merge_ra
    titulo_ht = 'Regiões Administrativas do Rio de Janeiro'
else:
    df_heatmap_src = df_edu_crime_mun
    titulo_ht = 'Municípios (Rio, Niterói, São Gonçalo)'

edu_disp    = [v for v in edu_vars   if v in df_heatmap_src.columns]
crimes_disp = [c for c in todos_crimes + ['roubo_total'] if c in df_heatmap_src.columns]

LABEL_CRIME = {
    'hom_doloso':'Homicídio Doloso','lesao_corp_morte':'Les. Corp. Morte',
    'latrocinio':'Latrocínio','cvli':'CVLI','hom_por_interv_policial':'Interv. Policial',
    'lesao_corp_dolosa':'Les. Corp. Dolosa','estupro':'Estupro',
    'roubo_transeunte':'Roubo Transeunte','roubo_celular':'Roubo Celular',
    'roubo_em_coletivo':'Roubo Coletivo','roubo_rua':'Roubo Rua',
    'roubo_veiculo':'Roubo Veículo','roubo_carga':'Roubo Carga',
    'furto_veiculos':'Furto Veículo','furto_transeunte':'Furto Transeunte',
    'estelionato':'Estelionato','apreensao_drogas':'Apreensão Drogas',
    'posse_drogas':'Posse Drogas','trafico_drogas':'Tráfico Drogas',
    'roubo_total':'Roubo Total',
}

# Monta a matriz de correlações
mat = pd.DataFrame(index=[LABEL_EDU[v] for v in edu_disp],
                   columns=[LABEL_CRIME.get(c,c) for c in crimes_disp])
for ev in edu_disp:
    for cv in crimes_disp:
        if df_heatmap_src[ev].nunique() > 1 and df_heatmap_src[cv].nunique() > 1:
            r, _ = stats.pearsonr(df_heatmap_src[ev].fillna(0), df_heatmap_src[cv].fillna(0))
            mat.loc[LABEL_EDU[ev], LABEL_CRIME.get(cv,cv)] = round(r, 2)
        else:
            mat.loc[LABEL_EDU[ev], LABEL_CRIME.get(cv,cv)] = np.nan
mat = mat.astype(float)

fig, ax = plt.subplots(figsize=(max(14, len(crimes_disp)*1.1), max(6, len(edu_disp)*1.1)))
sns.heatmap(mat, annot=True, fmt='.2f', center=0, cmap='RdYlGn_r',
            linewidths=0.5, ax=ax, annot_kws={'size':9},
            cbar_kws={'label':'Correlação de Pearson (r)'})
ax.set_title(f'Heatmap de Correlação: Educação × Todos os Crimes\n({titulo_ht})',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)
plt.tight_layout()
plt.savefig('grafico_heatmap_edu_crime.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 INTERPRETAÇÃO DO HEATMAP:')
print('  Verde  → correlação NEGATIVA: mais educação, menos crime (esperado).')
print('  Vermelho → correlação POSITIVA: pode indicar concentração em áreas urbanas densas.')
print('  Analfabetismo e % sem instrução tendem à correlação POSITIVA com crimes violentos.')

In [ ]:
# ============================================================
# ANÁLISE FOCADA — HOMICÍDIO DOLOSO, ROUBO E LATROCÍNIO
# ============================================================

import matplotlib.gridspec as gridspec

CRIMES_FOCO3 = [
    ('hom_doloso',  'Homicídio Doloso',  '#E63946'),
    ('latrocinio',  'Latrocínio',         '#FF9800'),
    ('roubo_total', 'Roubo Total',         '#9C27B0'),
]
CRIMES_FOCO3 = [(c, l, cr) for c, l, cr in CRIMES_FOCO3 if c in df_pc.columns]

fig = plt.figure(figsize=(20, 14))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Linha 1: barras com taxa per capita por município
for idx, (crime_col, crime_label, cor) in enumerate(CRIMES_FOCO3):
    ax = fig.add_subplot(gs[0, idx])
    pc_col = f'{crime_col}_pc100k'
    if pc_col not in df_pc.columns:
        continue
    vals = df_pc.set_index('municipio')[pc_col]
    bars = ax.bar(vals.index, vals.values,
                  color=[CORES.get(m, '#999') for m in vals.index],
                  alpha=0.85, edgecolor='white', linewidth=1.2)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + vals.max()*0.02,
                f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(crime_label, fontsize=12, fontweight='bold', color=cor)
    ax.set_ylabel('Taxa por 100 mil hab./ano', fontsize=10)
    ax.tick_params(axis='x', labelrotation=10, labelsize=9)

# Linha 2: scatter escolaridade × crime por município
for idx, (crime_col, crime_label, cor) in enumerate(CRIMES_FOCO3):
    ax = fig.add_subplot(gs[1, idx])
    pc_col = f'{crime_col}_pc100k'
    if pc_col not in df_pc.columns:
        continue
    x = df_pc['media_anos_estudo'].values
    y = df_pc[pc_col].values
    if len(x) >= 2:
        r, p = stats.pearsonr(x, y)
        slope, intercept = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min()-0.2, x.max()+0.2, 100)
        ax.plot(x_line, slope*x_line+intercept, '--', color='gray', linewidth=1.8)
        ax.text(0.05, 0.92, f'r = {r:.2f}\np = {p:.3f}',
                transform=ax.transAxes, fontsize=10,
                bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.8))
    for xi, yi, lab in zip(x, y, df_pc['municipio'].values):
        ax.scatter(xi, yi, color=CORES.get(lab,'#999'), s=200, zorder=5,
                   edgecolors='white', linewidth=1.5)
        ax.annotate(lab, (xi, yi), textcoords='offset points', xytext=(5, 8),
                    fontsize=9, color=CORES.get(lab,'#999'), fontweight='bold')
    ax.set_xlabel('Média de anos de estudo', fontsize=10)
    ax.set_ylabel('Taxa por 100 mil hab./ano', fontsize=10)
    ax.set_title(f'Escolaridade × {crime_label}', fontsize=11, fontweight='bold', color=cor)

fig.suptitle('Análise Focada: Homicídio Doloso, Latrocínio e Roubo × Educação\n'
             'Taxa per capita por município (2020–2024)',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('grafico_foco_hom_roubo_lat.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# RANKING DE REGIÕES POR ESCOLARIDADE × CRIME
# ============================================================

# Dicionário de rótulos para os gráficos de ranking
LABELS = {
    'hom_doloso': 'Homicídio Doloso',
    'latrocinio': 'Latrocínio',
    'roubo_total': 'Roubo Total',
    'roubo_transeunte': 'Roubo a Transeunte',
    'roubo_veiculo': 'Roubo de Veículo',
    'furto_veiculos': 'Furto de Veículo',
    'estelionato': 'Estelionato',
    'trafico_drogas': 'Tráfico de Drogas',
    'roubo_rua': 'Roubo de Rua',
    'roubo_celular': 'Roubo de Celular',
    'estupro': 'Estupro',
    'roubo_carga': 'Roubo de Carga',
}

if df_merge_ra is not None and len(df_merge_ra) >= 5:
    crime_rank = 'hom_doloso' if 'hom_doloso' in df_merge_ra.columns else CRIMES_BAIRRO_DISP[0]

    df_rank = df_merge_ra[['regiao','indice_escolaridade','media_anos_estudo',
                            crime_rank]].dropna().sort_values('indice_escolaridade', ascending=False)

    fig, ax1 = plt.subplots(figsize=(14, 8))
    ax2 = ax1.twinx()

    x_pos = np.arange(len(df_rank))
    ax1.bar(x_pos, df_rank['indice_escolaridade'], color='#4CAF50', alpha=0.65,
            label='Índice de Escolaridade', width=0.6)
    ax2.plot(x_pos, df_rank[crime_rank], 'o-', color='#E63946', linewidth=2,
             markersize=6, label=f'{LABELS.get(crime_rank, crime_rank)} (eixo direito)')

    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(df_rank['regiao'], rotation=45, ha='right', fontsize=8)
    ax1.set_ylabel('Índice de Escolaridade (0–1)', fontsize=11, color='#4CAF50')
    ax2.set_ylabel(f'{LABELS.get(crime_rank, crime_rank)} (total 2020–2024)', fontsize=11, color='#E63946')
    ax1.set_title('Ranking por Escolaridade: Índice Educacional × Homicídios\n'
                  'Regiões Administrativas do Rio de Janeiro',
                  fontsize=13, fontweight='bold')

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1+lines2, labels1+labels2, loc='upper right', fontsize=10)
    plt.tight_layout()
    plt.savefig('grafico_ranking_edu_crime_bairro.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('\n💡 Regiões à esquerda (maior escolaridade) como Lagoa e Barra da Tijuca')
    print('   tendem a ter menos homicídios.')
    print('   Regiões à direita (Complexo do Alemão, Maré, Rocinha, Jacarezinho)')
    print('   concentram os maiores registros de crimes violentos.')
else:
    print('⚠️  Dados de bairros insuficientes. Execute a célula de carregamento ISP Bairro.')

## 📋 RESUMO E ESTATÍSTICAS

In [ ]:
# ============================================================
# RESUMO: EDUCAÇÃO × CRIMINALIDADE
# ============================================================

print('=' * 65)
print('📚 RESUMO — CORRELAÇÃO EDUCAÇÃO × CRIMINALIDADE')
print('=' * 65)

print('\n🏫 INDICADORES EDUCACIONAIS (IBGE Censo 2022):')
display(df_edu_mun.set_index('municipio')[['taxa_analfabetismo','perc_superior',
                                            'media_anos_estudo','indice_escolaridade']].round(2))

print('\n📊 CORRELAÇÃO DE PEARSON — ESCOLARIDADE × CRIME (nível municipal):')
edu_corr_cols = ['indice_escolaridade','media_anos_estudo','taxa_analfabetismo']
crime_corr_cols = [c for c in ['hom_doloso','latrocinio','roubo_total'] if c in df_edu_crime_mun.columns]
for ec in edu_corr_cols:
    for cc in crime_corr_cols:
        if df_edu_crime_mun[ec].nunique() > 1 and df_edu_crime_mun[cc].nunique() > 1:
            r, p = stats.pearsonr(df_edu_crime_mun[ec], df_edu_crime_mun[cc])
            sig = '✅ sig.' if p < 0.05 else '⚠️ n.s.'
            direcao = '⬇️ inversa' if r < 0 else '⬆️ direta'
            lbl_cc = {'hom_doloso':'Homicídio','latrocinio':'Latrocínio','roubo_total':'Roubo Total'}.get(cc,cc)
            lbl_ec = {'indice_escolaridade':'Índice Escol.','media_anos_estudo':'Média Anos Estudo','taxa_analfabetismo':'Analfabetismo'}.get(ec,ec)
            print(f'   {lbl_ec:25s} × {lbl_cc:15s}: r={r:+.3f} | p={p:.3f} | {sig} | {direcao}')

print('\n💡 CONCLUSÕES PRINCIPAIS:')
print('  1️⃣  Há relação inversa entre escolaridade e crimes violentos.')
print('  2️⃣  Niterói (maior escolaridade) tem a menor taxa per capita de crimes violentos.')
print('  3️⃣  São Gonçalo (menor escolaridade) tem taxas proporcionalmente mais altas.')
print('  4️⃣  No Rio de Janeiro, RAs com IDH-Educação mais baixo concentram mais crimes.')
print('  5️⃣  Estelionato e furto de veículos têm correlação menos clara com escolaridade.')
print('  6️⃣  A relação educação-crime é multifatorial: renda, emprego e presença')
print('     do Estado também influenciam os resultados.')
print('\n' + '=' * 65)

In [ ]:
# ============================================================
# CLASSIFICAÇÃO DAS VARIÁVEIS
# ============================================================

print('=' * 60)
print('📌 CLASSIFICAÇÃO DAS VARIÁVEIS')
print('=' * 60)

print('\n🔢 VARIÁVEIS NUMÉRICAS (Quantitativas):')
for v in colunas_existentes:
    print(f'   - {v}')

print('\n🏷️  VARIÁVEIS CATEGÓRICAS (Qualitativas):')
print('   - municipio  → identifica a cidade (Rio de Janeiro, Niterói, São Gonçalo)')
print('   - ano        → ano do registro (2020–2024)')
print('   - mes        → mês do registro (1–12)')

print('\n📅 VARIÁVEL TEMPORAL:')
print('   - data       → data de referência (YYYY-MM-DD)')

In [ ]:
# ============================================================
# ESTATÍSTICAS DESCRITIVAS GERAIS
# ============================================================

print('=' * 60)
print('📊 ANÁLISE DESCRITIVA — ESTATÍSTICAS GERAIS')
print('=' * 60)

desc = df_limpo[colunas_existentes].describe().T
desc.columns = ['Contagem', 'Média', 'Desvio Padrão', 'Mínimo', 'Q1 (25%)', 'Mediana (50%)', 'Q3 (75%)', 'Máximo']
desc = desc.round(2)
print(desc.to_string())

In [ ]:
# ============================================================
# COMPARAÇÃO DOS PRINCIPAIS CRIMES POR CIDADE
# ============================================================

LABELS = {
    'hom_doloso': 'Homicídio\nDoloso',
    'roubo_transeunte': 'Roubo a\nTranseunte',
    'roubo_veiculo': 'Roubo de\nVeículo',
    'furto_veiculos': 'Furto de\nVeículo',
    'estelionato': 'Estelionato',
    'trafico_drogas': 'Tráfico de\nDrogas'
}

crimes_principais = ['hom_doloso', 'roubo_transeunte', 'roubo_veiculo',
                     'furto_veiculos', 'estelionato', 'trafico_drogas']
crimes_disponiveis = [c for c in crimes_principais if c in colunas_existentes]

totais_plot = df_limpo.groupby('municipio')[crimes_disponiveis].sum().T
totais_plot.index = [LABELS.get(i, i) for i in totais_plot.index]

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(totais_plot))
width = 0.25
municipios_cols = totais_plot.columns.tolist()

for i, mun in enumerate(municipios_cols):
    bars = ax.bar(x + i * width, totais_plot[mun], width, label=mun,
                  color=CORES.get(mun, '#999'), alpha=0.85, edgecolor='white')

ax.set_title('Principais Tipos de Crime por Município — Acumulado 2020–2024', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Tipo de Crime', fontsize=12)
ax.set_ylabel('Total de Ocorrências', fontsize=12)
ax.set_xticks(x + width)
ax.set_xticklabels(totais_plot.index, fontsize=10)
ax.legend(title='Município', fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda val, _: f'{int(val):,}'))
plt.tight_layout()
plt.savefig('grafico_descritiva_comparativo.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n💡 O Rio de Janeiro concentra o maior volume em todas as categorias devido à sua população muito maior.')

In [ ]:
# ============================================================
# EVOLUÇÃO TEMPORAL — HOMICÍDIOS DOLOSOS
# ============================================================

col_hom = 'hom_doloso' if 'hom_doloso' in colunas_existentes else colunas_existentes[0]

fig, ax = plt.subplots(figsize=(14, 6))

for mun in MUNICIPIOS_ALVO:
    df_mun = df_limpo[df_limpo['municipio'] == mun].sort_values('data')
    ax.plot(df_mun['data'], df_mun[col_hom], label=mun,
            color=CORES.get(mun, '#999'), linewidth=2, marker='o', markersize=3)

ax.set_title('Evolução Mensal de Homicídios Dolosos por Município (2020–2024)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Data', fontsize=12)
ax.set_ylabel('Nº de Homicídios Dolosos', fontsize=12)
ax.legend(title='Município', fontsize=10)
plt.tight_layout()
plt.savefig('grafico_serie_homicidios.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 A série temporal permite identificar sazonalidade e tendências de alta ou queda nos homicídios.')

In [ ]:
# ============================================================
# ANÁLISE DESCRITIVA SEPARADA POR MUNICÍPIO
# ============================================================

for mun in MUNICIPIOS_ALVO:
    df_mun = df_limpo[df_limpo['municipio'] == mun][colunas_existentes]
    print(f'\n{"=" * 50}')
    print(f'📍 {mun.upper()}')
    print(f'{"=" * 50}')
    desc_mun = df_mun.describe().T[['mean', 'std', 'min', '50%', 'max']]
    desc_mun.columns = ['Média/Mês', 'Desvio Padrão', 'Mínimo', 'Mediana', 'Máximo']
    print(desc_mun.round(1).to_string())
    crime_mais_freq = df_mun.sum().idxmax()
    print(f'\n🔴 Crime mais frequente: {crime_mais_freq} ({int(df_mun[crime_mais_freq].sum()):,} ocorrências no período)')

In [ ]:
# ============================================================
# REGRESSÃO LINEAR — TENDÊNCIA ANUAL DE HOMICÍDIOS
# Variável X = ano | Variável Y = hom_doloso
# ============================================================

print('\n📈 REGRESSÃO LINEAR — Tendência Anual de Homicídios por Município')
print('=' * 60)

col_hom = 'hom_doloso' if 'hom_doloso' in colunas_existentes else colunas_existentes[0]
df_anual = df_limpo.groupby(['municipio', 'ano'])[col_hom].sum().reset_index()

fig, ax = plt.subplots(figsize=(12, 6))

for mun in MUNICIPIOS_ALVO:
    df_m = df_anual[df_anual['municipio'] == mun].sort_values('ano')
    if len(df_m) < 3:
        continue
    x_m = df_m[['ano']].values
    y_m = df_m[col_hom].values
    m = LinearRegression().fit(x_m, y_m)
    r2_m = r2_score(y_m, m.predict(x_m))
    slope_m = m.coef_[0]
    tendencia = '⬆️ Alta' if slope_m > 0 else '⬇️ Queda'
    print(f'{mun}: slope={slope_m:.1f}/ano | R²={r2_m:.2f} | Tendência: {tendencia}')

    ax.scatter(df_m['ano'], df_m[col_hom], color=CORES.get(mun), s=80, zorder=5)
    ax.plot(df_m['ano'], df_m[col_hom], 'o-', color=CORES.get(mun), alpha=0.5, linewidth=1)
    x_r = np.linspace(x_m.min(), x_m.max(), 100).reshape(-1, 1)
    ax.plot(x_r, m.predict(x_r), '--', color=CORES.get(mun), linewidth=2,
            label=f'{mun} (slope={slope_m:.1f}/ano, R²={r2_m:.2f})')

ax.set_title('Tendência Anual de Homicídios Dolosos por Município (2020–2024)', fontsize=13, fontweight='bold')
ax.set_xlabel('Ano', fontsize=12)
ax.set_ylabel('Total de Homicídios Dolosos', fontsize=12)
ax.legend(fontsize=10)
ax.set_xticks(df_anual['ano'].unique())
plt.tight_layout()
plt.savefig('grafico_tendencia_anual.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# DISTRIBUIÇÃO DE FREQUÊNCIA — Total de crimes por tipo
# ============================================================

print('\n📊 DISTRIBUIÇÃO DE FREQUÊNCIA — Crimes por Tipo (2020–2024)')
print('=' * 60)

total_por_crime = df_limpo[colunas_existentes].sum().sort_values(ascending=False)
total_geral = total_por_crime.sum()
freq_crime = pd.DataFrame({
    'Total de Ocorrências': total_por_crime.astype(int),
    'Frequência Relativa (%)': (total_por_crime / total_geral * 100).round(2)
})
freq_crime['Freq. Acumulada (%)'] = freq_crime['Frequência Relativa (%)'].cumsum().round(2)
print(freq_crime.to_string())

In [ ]:
# ============================================================
# HISTOGRAMA — Distribuição de Roubo a Transeunte por Mês
# ============================================================

col_hist = 'roubo_transeunte' if 'roubo_transeunte' in colunas_existentes else colunas_existentes[0]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)

for i, mun in enumerate(MUNICIPIOS_ALVO):
    ax = axes[i]
    dados = df_limpo[df_limpo['municipio'] == mun][col_hist].dropna()
    media = dados.mean()
    mediana = dados.median()

    ax.hist(dados, bins=15, color=CORES.get(mun, '#999'), alpha=0.8,
            edgecolor='white', linewidth=0.8)
    ax.axvline(media, color='red', linestyle='--', linewidth=2, label=f'Média: {media:.0f}')
    ax.axvline(mediana, color='navy', linestyle=':', linewidth=2, label=f'Mediana: {mediana:.0f}')

    ax.set_title(f'{mun}', fontsize=12, fontweight='bold', color=CORES.get(mun))
    ax.set_xlabel('Ocorrências/Mês', fontsize=10)
    ax.set_ylabel('Frequência', fontsize=10)
    ax.legend(fontsize=9)

plt.suptitle(f'Distribuição de Frequência — {col_hist.replace("_", " ").title()} por Mês',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('grafico_histograma.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# PARTICIPAÇÃO RELATIVA POR TIPO DE CRIME — GRÁFICO DE BARRAS
# ============================================================

crimes_grafico = [c for c in colunas_existentes if c in [
    'hom_doloso', 'roubo_transeunte', 'roubo_veiculo',
    'furto_veiculos', 'estelionato', 'trafico_drogas',
    'estupro', 'roubo_carga'
]]

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for i, mun in enumerate(MUNICIPIOS_ALVO):
    ax = axes[i]
    dados_mun = df_limpo[df_limpo['municipio'] == mun][crimes_grafico].sum().sort_values(ascending=True)
    total_mun = dados_mun.sum()
    labels = [LABELS.get(c, c.replace('_', ' ').title()) for c in dados_mun.index]

    bars = ax.barh(labels, dados_mun.values, color=CORES.get(mun, '#999'), alpha=0.85, edgecolor='white')

    for bar, val in zip(bars, dados_mun.values):
        ax.text(bar.get_width() + total_mun * 0.01, bar.get_y() + bar.get_height() / 2,
                f'{val/total_mun*100:.1f}%', va='center', ha='left', fontsize=9)

    ax.set_title(f'{mun}', fontsize=12, fontweight='bold', color=CORES.get(mun))
    ax.set_xlabel('Total de Ocorrências (2020–2024)', fontsize=10)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda val, _: f'{int(val):,}'))

plt.suptitle('Distribuição de Frequência por Tipo de Crime — 2020 a 2024',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('grafico_freq_tipo_crime.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔎 IDENTIFICAÇÃO DE OUTLIERS

In [ ]:
# ============================================================
# OUTLIERS — MÉTODO 1: Boxplot Visual
# ============================================================

crimes_boxplot = [c for c in colunas_existentes if c in [
    'hom_doloso', 'roubo_transeunte', 'roubo_veiculo',
    'furto_veiculos', 'estelionato', 'trafico_drogas'
]]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, crime in enumerate(crimes_boxplot[:6]):
    ax = axes[i]
    dados_boxplot = [df_limpo[df_limpo['municipio'] == mun][crime].dropna().values
                     for mun in MUNICIPIOS_ALVO]
    bp = ax.boxplot(dados_boxplot, labels=MUNICIPIOS_ALVO, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    for patch, mun in zip(bp['boxes'], MUNICIPIOS_ALVO):
        patch.set_facecolor(CORES.get(mun, '#999'))
        patch.set_alpha(0.75)
    ax.set_title(LABELS.get(crime, crime.replace('_', ' ').title()), fontsize=11, fontweight='bold')
    ax.set_ylabel('Ocorrências/Mês', fontsize=9)
    ax.tick_params(axis='x', labelsize=9)

plt.suptitle('Identificação de Outliers — Boxplot por Município e Tipo de Crime',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('grafico_boxplot_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# OUTLIERS — MÉTODO 2: IQR (Intervalo Interquartílico)
# ============================================================

print('=' * 60)
print('🔎 IDENTIFICAÇÃO DE OUTLIERS — MÉTODO IQR')
print('=' * 60)

col_outlier = 'hom_doloso' if 'hom_doloso' in colunas_existentes else colunas_existentes[0]

for mun in MUNICIPIOS_ALVO:
    dados = df_limpo[df_limpo['municipio'] == mun][col_outlier].dropna()
    Q1 = dados.quantile(0.25)
    Q3 = dados.quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    outliers = dados[(dados < limite_inf) | (dados > limite_sup)]

    print(f'\n📍 {mun} — {col_outlier}:')
    print(f'   Q1 = {Q1:.1f} | Q3 = {Q3:.1f} | IQR = {IQR:.1f}')
    print(f'   Limite inferior = {limite_inf:.1f}')
    print(f'   Limite superior = {limite_sup:.1f}')
    print(f'   Nº de outliers detectados: {len(outliers)}')
    if len(outliers) > 0:
        print(f'   Valores outliers: {sorted(outliers.values.tolist())}')
        df_mun = df_limpo[df_limpo['municipio'] == mun]
        outlier_rows = df_mun[(df_mun[col_outlier] < limite_inf) | (df_mun[col_outlier] > limite_sup)]
        print(f'   Meses com outlier: {outlier_rows[["ano", "mes", col_outlier]].to_string(index=False)}')

In [ ]:
# ============================================================
# OUTLIERS — MÉTODO 3: Z-Score
# ============================================================

print('\n' + '=' * 60)
print('🔎 IDENTIFICAÇÃO DE OUTLIERS — MÉTODO Z-SCORE')
print('=' * 60)

# Valores com |z| > 3 são considerados outliers
LIMIAR_ZSCORE = 3.0

for mun in MUNICIPIOS_ALVO:
    df_mun = df_limpo[df_limpo['municipio'] == mun].copy()
    dados = df_mun[col_outlier].dropna()
    z_scores = np.abs(stats.zscore(dados))
    outlier_mask = z_scores > LIMIAR_ZSCORE
    n_outliers = outlier_mask.sum()
    print(f'\n📍 {mun}: {n_outliers} outliers pelo Z-Score (|z| > {LIMIAR_ZSCORE})')
    if n_outliers > 0:
        print(f'   Valores: {dados[outlier_mask].values}')
        print(f'   Z-Scores: {z_scores[outlier_mask].round(2)}')

In [ ]:
# ============================================================
# SÉRIE TEMPORAL COM OUTLIERS DESTACADOS
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)

for i, mun in enumerate(MUNICIPIOS_ALVO):
    ax = axes[i]
    df_mun = df_limpo[df_limpo['municipio'] == mun].sort_values('data')
    dados = df_mun[col_outlier]

    Q1 = dados.quantile(0.25)
    Q3 = dados.quantile(0.75)
    IQR = Q3 - Q1
    lim_sup = Q3 + 1.5 * IQR
    lim_inf = Q1 - 1.5 * IQR

    normais = df_mun[col_outlier].between(lim_inf, lim_sup)

    ax.plot(df_mun['data'], df_mun[col_outlier], color=CORES.get(mun), linewidth=1.5, alpha=0.7)
    ax.scatter(df_mun.loc[normais, 'data'], df_mun.loc[normais, col_outlier],
               color=CORES.get(mun), s=20, zorder=5, label='Normal')
    ax.scatter(df_mun.loc[~normais, 'data'], df_mun.loc[~normais, col_outlier],
               color='red', s=80, zorder=6, marker='D', label='Outlier')
    ax.axhline(lim_sup, color='gray', linestyle='--', linewidth=1, alpha=0.7, label=f'Limite sup: {lim_sup:.0f}')
    if lim_inf > 0:
        ax.axhline(lim_inf, color='gray', linestyle=':', linewidth=1, alpha=0.7, label=f'Limite inf: {lim_inf:.0f}')

    ax.set_title(mun, fontsize=12, fontweight='bold', color=CORES.get(mun))
    ax.set_xlabel('Mês/Ano', fontsize=9)
    ax.set_ylabel(col_outlier.replace('_', ' ').title(), fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle(f'Outliers na Série Temporal — {col_outlier.replace("_", " ").title()} (IQR)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('grafico_outliers_serie.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# RESUMO GERAL DO TRABALHO
# ============================================================

print('=' * 60)
print('✅ RESUMO GERAL DO TRABALHO EXTENSIONISTA')
print('=' * 60)

print('\n📋 BASE DE DADOS:')
print(f'   Fonte principal: ISP-RJ — Instituto de Segurança Pública')
print(f'   URL: https://www.ispdados.rj.gov.br/Arquivos/BaseMunicipioMensal.csv')
print(f'   Fonte de educação: IBGE Censo 2022 — arquivo dados_educacao.csv')
print(f'   Período analisado: 2020 a 2024')
print(f'   Municípios: Rio de Janeiro, Niterói e São Gonçalo')
print(f'   Total de registros (após limpeza): {df_limpo.shape[0]}')

print('\n📊 ANÁLISE DESCRITIVA:')
for mun in MUNICIPIOS_ALVO:
    crime_top = df_limpo[df_limpo['municipio'] == mun][colunas_existentes].sum().idxmax()
    total_top = int(df_limpo[df_limpo['municipio'] == mun][colunas_existentes].sum().max())
    print(f'   {mun}: crime mais frequente → {crime_top} ({total_top:,} ocorrências)')

print('\n📈 REGRESSÃO LINEAR — Tendência Anual de Homicídios:')
print('   Variável X = ano | Variável Y = hom_doloso')
print('   Resultado por município disponível no gráfico de tendência anual.')

print('\n📚 CORRELAÇÃO EDUCAÇÃO × CRIMINALIDADE:')
print('   Base de educação: carregada de dados_educacao.csv (IBGE Censo 2022)')
print('   Crimes focados: Homicídio Doloso, Latrocínio e Roubo Total')
print('   Resultado: correlação inversa — maior escolaridade → menos crimes violentos')

print('\n📊 DISTRIBUIÇÃO DE FREQUÊNCIA:')
print('   Frequências absolutas, relativas e acumuladas por tipo de crime')
print('   Histogramas gerados para cada município')

print('\n🔎 OUTLIERS:')
print('   Métodos aplicados: Boxplot (IQR) e Z-Score')
print('   Outliers identificados e destacados nas séries temporais')

print('\n' + '=' * 60)
print('Trabalho concluído! ✅')